# 探索数据流
从.pt文件到模型输入，逐步看每一步的shape和内容

In [1]:
import sys
sys.path.insert(0, '/home/PuMengYu/MSD_LiverTumorSeg/medseg_project')
sys.path.insert(0, '/home/PuMengYu/MSD_LiverTumorSeg/twostage_medseg')

import torch
import numpy as np
import matplotlib.pyplot as plt

print('torch:', torch.__version__)
print('cuda:', torch.cuda.is_available())

torch: 2.3.0+cu121
cuda: True


## 1. 直接加载一个.pt文件，看原始数据

In [ ]:
pt_path = '/home/PuMengYu/MSD_LiverTumorSeg/Task03_Liver_roi/liver_75.pt'
data = torch.load(pt_path, map_location='cpu')

# 看看pt里有哪些key
print('keys:', list(data.keys()))
for k, v in data.items():
    if isinstance(v, torch.Tensor):
        print(f'  {k}: shape={v.shape}, dtype={v.dtype}, min={v.min():.3f}, max={v.max():.3f}')
    else:
        print(f'  {k}: {v}')

## 2. 可视化：看image和label的某个slice

In [ ]:
image = data['image']  # 根据上面的key改
label = data['label']

# 取中间一个slice
z = image.shape[-1] // 2

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(image[0, :, :, z].numpy(), cmap='gray')
axes[0].set_title(f'image slice z={z}')
axes[1].imshow(label[0, :, :, z].numpy(), cmap='jet', vmin=0, vmax=2)
axes[1].set_title(f'label slice z={z}  (0=bg, 1=liver, 2=tumor)')
plt.tight_layout()
plt.show()

## 3. 过一遍TumorROIDataset，看__getitem__输出

In [ ]:
from medseg_project.medseg.data.dataset_offline import load_pt_paths, split_two_with_monitor
from twostage_medseg.twostage.dataset_tumor_roi import TumorROIDataset

all_pt = load_pt_paths('/home/PuMengYu/MSD_LiverTumorSeg/Task03_Liver_roi')
print('总样本数:', len(all_pt))

train_files, val_files = split_two_with_monitor(all_pt, val_ratio=0.2, seed=42)
print('train:', len(train_files), 'val:', len(val_files))

In [ ]:
# 构建Dataset，不用coarse tumor，先看基础版本
val_ds = TumorROIDataset(
    pt_paths=val_files[:3],   # 只取3个，快一点
    patch_size=(160, 160, 160),
    margin=12,
    is_train=False,
    use_coarse_tumor=False,
)

sample = val_ds[0]
print('sample keys:', list(sample.keys()))
for k, v in sample.items():
    if isinstance(v, torch.Tensor):
        print(f'  {k}: shape={v.shape}, dtype={v.dtype}')

## 4. 过一遍模型forward，看输出shape

In [ ]:
from medseg_project.medseg.models.build_model import build_model

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

model = build_model(
    model_name='dynunet_deep',
    in_channels=1,
    out_channels=1,
    patch_size=(160, 160, 160),
).to(device)

# 构造一个假输入，batch=1
x = torch.randn(1, 1, 160, 160, 160).to(device)
with torch.no_grad():
    out = model(x)

if isinstance(out, (list, tuple)):
    print('输出是列表（deep supervision）:')
    for i, o in enumerate(out):
        print(f'  out[{i}]: shape={o.shape}')
else:
    print('输出shape:', out.shape)